# Awkward Array ― 处理形状不整齐的数据

## 本章要学什么

NumPy 的数组以「每一行长度都相同」的方形数据为前提，而实验数据或 JSON 那样每行元素个数不同的数据，就没法直接表示。Awkward Array 就是为了让这类数据仍然以数组的形式被处理而出现的库。

- 生成每行长度不同的数组(不规则数组)的方法
- 缺失值的处理(值的缺失与整行的缺失)
- 类型混杂的数组(union 型)
- 字符串的拆分与转换成记录
- 深层嵌套 JSON 的汇总

请注意，整个过程中都不必写 Python 的循环，计算始终在数组上进行。

先导入需要的模块，其中 `display()` 用来把数组的内容以 HTML 显示出来。

In [1]:
from pathlib import Path

import awkward as ak
import numpy as np
from IPython.display import display

## 大小不整齐的数据

NumPy 的数组以方形为前提，因此无法表示每行元素个数不同的数据。下面的代码给 NumPy 的数组加上条件来选元素，结果被压成了一维，已经看不出某个元素原来属于哪一行。

In [2]:
np.random.seed(42)
matrix = np.random.randint(0, 10, size=(3, 10))
matrix[matrix > 5]

array([6, 7, 6, 9, 6, 7, 7, 7, 7, 9, 8, 9, 6], dtype=int32)

转换成 Awkward Array 之后，同样可以用条件选元素，而且能在保留行的结构的同时把计算继续做下去。

In [3]:
ak_matrix = ak.Array(matrix)
display(ak_matrix)
display(ak_matrix[ak_matrix > 5])

<Array [[6, 3, 7, 4, 6, 9, 2, 6, 7, 4], ..., [4, ..., 3]] type='3 * 10 * int32'>

<Array [6, 7, 6, 9, 6, 7, 7, 7, 7, 9, 8, 9, 6] type='13 * int32'>

要让每行的长度不同，就用 `ak.from_regular()` 把固定长的轴变成可变长的轴。

In [4]:
ragged = ak.from_regular(ak_matrix)
ragged

<Array [[6, 3, 7, 4, 6, 9, 2, 6, 7, 4], ..., [4, ...]] type='3 * var * int32'>

对这种不规则数组加条件，只是让每行剩下的元素个数不同而已，计算照样可以进行。

In [5]:
ragged_filtered = ragged[ragged > 5]
ragged_filtered

<Array [[6, 7, 6, 9, 6, 7], [7, ...], [9, 8, 9, 6]] type='3 * var * int32'>

`sum()` 这类汇总也和 NumPy 一样，指定 `axis` 就能做。

In [6]:
np.sum(ragged_filtered, axis=1)   # 每行的合计

<Array [41, 21, 32] type='3 * int64'>

In [7]:
np.sum(ragged_filtered, axis=0)   # 按位置的合计(因为不规则，长度按最长的行补)

<Array [22, 22, 22, 15, 6, 7] type='6 * int64'>

由于 Awkward Array 对不规则数据也能以数组的形式计算，所以完全不需要写 Python 循环。在这个环境里，下面这个例子的执行时间在 NumPy 版约为 3.8 秒，而 Awkward 版约为 0.06 秒。例子把一维数组按「值变小的位置」切开，再求每个区间的平均，先看 NumPy 的写法。

In [8]:
num_values = 1000000
values = np.random.randint(0, 10, num_values, dtype=np.int16)
split_points = np.where(np.diff(values) < 0)[0] + 1   # 值变小的位置

In [9]:
%%time
parts_numpy = np.split(values, split_points)
means_numpy = np.array([np.mean(part) for part in parts_numpy])

CPU times: total: 3.02 s
Wall time: 3.13 s


再看同样计算的 Awkward 写法：用 `unflatten()` 按区间切分，再用 `mean()` 一次求出平均，整个过程没有 Python 循环。

In [10]:
%%time
lengths = np.diff(split_points, prepend=0, append=len(values))
parts_awkward = ak.unflatten(values, lengths)
means_awkward = ak.mean(parts_awkward, axis=1)

CPU times: total: 78.1 ms
Wall time: 88.2 ms


确认两个结果相同。

In [11]:
np.all(means_numpy == means_awkward)

np.True_

把最前面的 10 个区间并排看一看，就能确认 NumPy 与 Awkward 表示的是同一份数据。

In [12]:
parts_numpy[:10]

[array([8, 8], dtype=int16),
 array([2, 4, 4, 5], dtype=int16),
 array([2, 8], dtype=int16),
 array([6, 8], dtype=int16),
 array([4, 7, 8], dtype=int16),
 array([3, 6, 7], dtype=int16),
 array([1, 5], dtype=int16),
 array([3, 8], dtype=int16),
 array([4], dtype=int16),
 array([3, 3], dtype=int16)]

In [13]:
parts_awkward[:10]

<Array [[8, 8], [2, 4, 4, 5], [...], ..., [4], [3, 3]] type='10 * var * int16'>

## 含有缺失的数据

NumPy 里有 `masked_array` 这套表示缺失的机制。

In [14]:
values = np.arange(10)
masked = np.ma.masked_array(values, values % 2 == 0)
display(masked)
np.sum(masked)

masked_array(data=[--, 1, --, 3, --, 5, --, 7, --, 9],
             mask=[ True, False,  True, False,  True, False,  True, False,
                    True, False],
       fill_value=999999)

np.int64(25)

Awkward 也能直接处理带缺失的数组，用 `mask` 把满足条件的元素设为缺失。

In [15]:
values_ak = ak.Array(values)
masked_ak = values_ak.mask[values_ak % 2 != 0]
display(masked_ak)
ak.sum(masked_ak)

<Array [None, 1, None, 3, None, 5, None, 7, None, 9] type='10 * ?int64'>

np.int64(25)

Awkward 不只处理值的缺失，还能处理整行的缺失：下面的例子除了值的缺失之外，还造出了整行为 `None` 的情形。

In [16]:
np.random.seed(2)
grid = ak.Array(np.random.randint(0, 10, size=(6, 9)))
grid = grid.mask[grid >= 3]                    # 把 3 以上的值设为缺失
grid = grid.mask[ak.count(grid, axis=1) >= 7]  # 把剩余值有 7 个以上的行设为缺失
ak.mean(grid, axis=1)

<Array [None, 4.78, 5.5, None, 7.62, None] type='6 * ?float64'>

这样一来，即使数据里含有缺失、结构又复杂，平均之类的计算照样能直接做。

## 类型混杂的数据

Awkward Array 可以把不同类型的元素放进同一个数组，这称为 union 型(union type)，于是 NumPy 难以处理的混合类型数据也能对付。

In [17]:
arr = ak.Array([1.0, 2.0, "34 ms", "23.4 ms", 1.2])
arr

<Array [1, 2, '34 ms', '23.4 ms', 1.2] type='5 * union[float64, string]'>

从字符串里去掉 `" ms"` 再统一转换成数值，也只是一步的事。

In [18]:
ak.strings_astype(ak.str.replace_substring(arr, " ms", ""), np.float64)

<Array [1, 2, 34, 23.4, 1.2] type='5 * float64'>

## 字符串的处理

接下来看一个把一大段字符串转换成逐行字典的例子。

In [19]:
params_text = """pos_x: 1
pos_y: 20
pos_z: 3000
v_x: 2
v_y: 10
v_z: 230"""

In [20]:
records = [
    dict(name=name, value=float(value))
    for name, value in [line.split(":") for line in params_text.split("\n")]
]
records

[{'name': 'pos_x', 'value': 1.0},
 {'name': 'pos_y', 'value': 20.0},
 {'name': 'pos_z', 'value': 3000.0},
 {'name': 'v_x', 'value': 2.0},
 {'name': 'v_y', 'value': 10.0},
 {'name': 'v_z', 'value': 230.0}]

用 NumPy 的 `char.split` 做同样的事，得到的会是一堆 Python 对象的数组(dtype=object)。

In [21]:
text_array = np.array([params_text])
split_lines = np.char.split(text_array, "\n")
print(repr(text_array))
print(repr(split_lines))

array(['pos_x: 1\npos_y: 20\npos_z: 3000\nv_x: 2\nv_y: 10\nv_z: 230'],
      dtype='<U54')
array([list(['pos_x: 1', 'pos_y: 20', 'pos_z: 3000', 'v_x: 2', 'v_y: 10', 'v_z: 230'])],
      dtype=object)


而 Awkward Array 可以把字符串拆开，并原样处理嵌套的列表结构。

In [22]:
text_ak = ak.Array([params_text])
split_lines_ak = ak.str.split_pattern(text_ak, "\n")[0]
split_fields_ak = ak.str.split_pattern(split_lines_ak, ":")
display(text_ak)
display(split_lines_ak)
display(split_fields_ak)

<Array ['pos_x: 1\npos_y: 20\npos_z: 3000\nv_x: 2\nv_y: 10\nv_z: 230'] type='...'>

<Array ['pos_x: 1', 'pos_y: 20', ..., 'v_y: 10', 'v_z: 230'] type='6 * string'>

<Array [['pos_x', ' 1'], [...], ..., ['v_z', ' 230']] type='6 * var * string'>

接着把它转换成结构化数据(记录)。

In [23]:
items = ak.zip(
    {
        "name": split_fields_ak[:, 0],
        "value": ak.strings_astype(split_fields_ak[:, 1], np.float64),
    }
)
items

<Array [{name: 'pos_x', value: 1}, ..., {...}] type='6 * {name: string, val...'>

把 `name` 以 `"pos"` 开头的那些值取出来并展开。

In [24]:
x, y, z = items[ak.str.starts_with(items.name, "pos"), "value"]
print(x, y, z)

1.0 20.0 3000.0


## 深层嵌套的 JSON

最后看一个处理真实深层嵌套 JSON 的例子。

In [25]:
fn = r"data/employees_1MB.json"
employees = ak.from_json(Path(fn))
employees.type.show()

566 * {
    employee: {
        id: string,
        name: string,
        position: string,
        department: {
            id: string,
            name: string,
            manager: {
                id: string,
                name: string,
                contact: {
                    email: string,
                    phone: string
                }
            }
        },
        projects: var * {
            projectId: string,
            projectName: string,
            startDate: string,
            tasks: var * {
                taskId: string,
                title: string,
                status: string,
                details: {
                    hoursSpent: int64,
                    technologiesUsed: var * string,
                    completionDate: ?string,
                    expectedCompletion: ?string
                }
            }
        }
    }
}


这份 JSON 按员工 → 项目 → 任务 → 技术这样嵌套了 4 层以上，而 Awkward Array 可以把它直接当作数组来处理，例如要取出所有的 `technologiesUsed` 并统计各自被用到的次数，就可以像下面这样写。

In [26]:
tech = ak.ravel(employees["employee", "projects", "tasks", "details", "technologiesUsed"])
tech_sorted = ak.sort(tech)
tech_grouped = ak.unflatten(tech_sorted, ak.run_lengths(tech_sorted))

ak.zip({
    "name": tech_grouped[:, 0],
    "count": ak.num(tech_grouped, axis=1),
})

<Array [{name: 'Java', count: 356}, ..., {...}] type='10 * {name: string, c...'>

## 小结

Awkward Array 最能发挥作用的，是下面这类数据。

* 每行元素个数不同的不规则数据
* 含有缺失的数据(值的缺失与行的缺失)
* 类型混杂的 union 型数据
* 字符串处理与记录结构的处理
* 深层嵌套的 JSON

即使是 NumPy 的方形数组无法处理的数据，也仍然能以数组的形式做向量化计算。